# 04 - 使用预训练模型与微调


真实项目很少从零训练大模型，更多时候会使用 Hugging Face 上的预训练模型，并通过提示、微调或 LoRA 适配任务。本 notebook 在原教程基础上补充 tokenizer、pipeline、微调和 LoRA 的实践理解。

学习目标：

- 了解 Hugging Face transformers、datasets 和 Model Hub 的角色。
- 理解 tokenizer 为什么是文本模型的入口。
- 区分直接推理、全量微调、冻结层、LoRA 和 Prompt Tuning。
- 看懂 LoRA 为什么能用少量参数适配大模型。
- 建立从基础模型到实际项目的学习路线。

## 环境准备与导入

这一段保留原教程的导入、标题打印和基础设置。先运行它，后续代码单元会复用这里导入的库、函数和随机种子。

In [2]:
"""
第五章 5.4：使用预训练模型 (Hugging Face)
=========================================

在实际工作中，我们很少从零训练大模型。
而是使用预训练好的模型进行微调或直接推理。

Hugging Face 是最流行的预训练模型平台。

本节内容：
1. Hugging Face 生态介绍
2. Tokenizer 详解
3. 使用预训练模型做文本分类
4. 文本生成
5. 微调 (Fine-tuning) 基础
"""

import numpy as np
import torch
import torch.nn as nn

print("=" * 60)
print("第五章 5.4：使用预训练模型 (Hugging Face)")
print("=" * 60)

第五章 5.4：使用预训练模型 (Hugging Face)


## 1. Hugging Face 生态


Hugging Face 的价值在于统一接口：无论底层模型是 BERT、GPT、T5 还是 LLaMA，都可以通过 `AutoTokenizer`、`AutoModel` 等 API 加载。

初学时先理解 pipeline，再理解 tokenizer/model 的底层调用，会更容易建立完整心智模型。

In [3]:
print("\n" + "=" * 60)
print("1. Hugging Face 生态介绍")
print("=" * 60)

print("""
【Hugging Face 核心组件】

1. transformers 库:
   - 提供数千个预训练模型
   - 统一的 API: AutoModel, AutoTokenizer
   - 支持 PyTorch 和 TensorFlow

2. datasets 库:
   - 大量现成的数据集
   - 高效的数据加载

3. Model Hub (huggingface.co):
   - 社区分享的模型
   - 模型卡片、使用示例

【常用模型家族】
- BERT: 双向编码器，擅长理解任务
- GPT-2/3: 单向解码器，擅长生成
- T5: 编码器-解码器，万能模型
- LLaMA: Meta 的开源大模型

【使用方式】
  from transformers import AutoTokenizer, AutoModel
  
  tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
  model = AutoModel.from_pretrained("bert-base-uncased")
""")


1. Hugging Face 生态介绍

【Hugging Face 核心组件】

1. transformers 库:
   - 提供数千个预训练模型
   - 统一的 API: AutoModel, AutoTokenizer
   - 支持 PyTorch 和 TensorFlow

2. datasets 库:
   - 大量现成的数据集
   - 高效的数据加载

3. Model Hub (huggingface.co):
   - 社区分享的模型
   - 模型卡片、使用示例

【常用模型家族】
- BERT: 双向编码器，擅长理解任务
- GPT-2/3: 单向解码器，擅长生成
- T5: 编码器-解码器，万能模型
- LLaMA: Meta 的开源大模型

【使用方式】
  from transformers import AutoTokenizer, AutoModel

  tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
  model = AutoModel.from_pretrained("bert-base-uncased")



## 2. Tokenizer 详解


Tokenizer 决定文字如何变成 token id。子词分词是现代模型的主流，因为它在词表大小和未登录词处理之间取得平衡。

BPE 的直觉是反复合并高频相邻片段，让常见词或词根成为更长 token，同时保留拆解陌生词的能力。

In [4]:
print("\n" + "=" * 60)
print("2. Tokenizer (分词器) 详解")
print("=" * 60)

print("""
【为什么需要 Tokenizer？】
模型只能处理数字，不能直接处理文字。
Tokenizer 的任务: 文本 → 数字序列

【分词策略的演变】
1. 按词分词: "I love AI" → ["I", "love", "AI"]
   问题: 词表太大，无法处理未见过的词

2. 按字符分词: "love" → ["l", "o", "v", "e"]
   问题: 序列太长，丢失词级信息

3. 子词分词 (Subword): 当前主流！
   "unhappiness" → ["un", "happiness"]
   "playing" → ["play", "ing"]
   
   BPE (Byte Pair Encoding): GPT 使用
   WordPiece: BERT 使用
   SentencePiece: LLaMA/T5 使用
""")

# 简单的 BPE Tokenizer 演示
print("\n--- 简单 BPE 演示 ---")

class SimpleBPE:
    """简化版 BPE Tokenizer"""
    
    def __init__(self):
        self.vocab = {}
        self.merges = []
    
    def train(self, text, num_merges=10):
        """训练 BPE"""
        # 初始化: 每个字符是一个 token
        words = text.split()
        # 用空格分隔字符，加上 </w> 表示词尾
        word_freqs = {}
        for word in words:
            chars = ' '.join(list(word)) + ' </w>'
            word_freqs[chars] = word_freqs.get(chars, 0) + 1
        
        print(f"初始词表:")
        for word, freq in list(word_freqs.items())[:5]:
            print(f"  '{word}': {freq}")
        
        for i in range(num_merges):
            # 统计所有相邻 pair 的频率
            pairs = {}
            for word, freq in word_freqs.items():
                symbols = word.split()
                for j in range(len(symbols) - 1):
                    pair = (symbols[j], symbols[j+1])
                    pairs[pair] = pairs.get(pair, 0) + freq
            
            if not pairs:
                break
            
            # 找最频繁的 pair
            best_pair = max(pairs, key=pairs.get)
            self.merges.append(best_pair)
            
            # 合并该 pair
            new_word_freqs = {}
            bigram = ' '.join(best_pair)
            replacement = ''.join(best_pair)
            
            for word, freq in word_freqs.items():
                new_word = word.replace(bigram, replacement)
                new_word_freqs[new_word] = freq
            
            word_freqs = new_word_freqs
            
            if i < 5:
                print(f"\n合并 #{i+1}: '{best_pair[0]}' + '{best_pair[1]}' → '{replacement}'")
        
        print(f"\n最终词表示例:")
        for word, freq in list(word_freqs.items())[:5]:
            print(f"  '{word}': {freq}")
        
        return word_freqs

# 训练 BPE
text = "low lower lowest new newer newest"
text = (text + " ") * 10  # 重复增加频率
print(f"训练文本: 'low lower lowest new newer newest' (重复10次)")
bpe = SimpleBPE()
result = bpe.train(text, num_merges=8)

print(f"\n【关键理解】")
print(f"BPE 通过反复合并最频繁的相邻字符对来构建子词词表")
print(f"'low', 'lower', 'lowest' 共享前缀 'low'")
print(f"→ 模型可以利用子词之间的共享语义！")


2. Tokenizer (分词器) 详解

【为什么需要 Tokenizer？】
模型只能处理数字，不能直接处理文字。
Tokenizer 的任务: 文本 → 数字序列

【分词策略的演变】
1. 按词分词: "I love AI" → ["I", "love", "AI"]
   问题: 词表太大，无法处理未见过的词

2. 按字符分词: "love" → ["l", "o", "v", "e"]
   问题: 序列太长，丢失词级信息

3. 子词分词 (Subword): 当前主流！
   "unhappiness" → ["un", "happiness"]
   "playing" → ["play", "ing"]

   BPE (Byte Pair Encoding): GPT 使用
   WordPiece: BERT 使用
   SentencePiece: LLaMA/T5 使用


--- 简单 BPE 演示 ---
训练文本: 'low lower lowest new newer newest' (重复10次)
初始词表:
  'l o w </w>': 10
  'l o w e r </w>': 10
  'l o w e s t </w>': 10
  'n e w </w>': 10
  'n e w e r </w>': 10

合并 #1: 'w' + 'e' → 'we'

合并 #2: 'l' + 'o' → 'lo'

合并 #3: 'n' + 'e' → 'ne'

合并 #4: 'w' + '</w>' → 'w</w>'

合并 #5: 'lo' + 'we' → 'lowe'

最终词表示例:
  'lo w</w>': 10
  'lowe r</w>': 10
  'lowe st</w>': 10
  'ne w</w>': 10
  'ne we r</w>': 10

【关键理解】
BPE 通过反复合并最频繁的相邻字符对来构建子词词表
'low', 'lower', 'lowest' 共享前缀 'low'
→ 模型可以利用子词之间的共享语义！


## 3. 使用预训练模型


pipeline 是最快上手方式，适合验证模型能力；`AutoTokenizer` + `AutoModelFor...` 则适合需要控制输入、输出和生成参数的场景。

实际项目中要关注模型大小、许可证、语言能力、上下文长度、推理速度和部署成本。

In [5]:
print("\n" + "=" * 60)
print("3. 使用预训练模型 (代码示例)")
print("=" * 60)

print("""
【注意】以下代码需要安装 transformers 库并下载模型
如果网络环境允许，可以实际运行

--- 文本分类 (情感分析) ---

from transformers import pipeline

# 最简单的方式: 使用 pipeline
classifier = pipeline("sentiment-analysis")
result = classifier("I love this tutorial! It's amazing.")
print(result)
# [{'label': 'POSITIVE', 'score': 0.9998}]

--- 文本生成 ---

from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")
result = generator("Once upon a time", max_length=50)
print(result[0]['generated_text'])

--- 更细粒度的控制 ---

from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

# 编码
input_ids = tokenizer.encode("Hello, how are", return_tensors="pt")

# 生成
output = model.generate(input_ids, max_length=20, temperature=0.7)

# 解码
text = tokenizer.decode(output[0])
print(text)
""")

# 我们可以用之前实现的 MiniGPT 来模拟这个流程
print("\n--- 用我们的 MiniGPT 模拟 Hugging Face 的使用流程 ---")

# 模拟一个简单的 Tokenizer
class SimpleTokenizer:
    """模拟 Hugging Face Tokenizer 的接口"""
    
    def __init__(self, vocab):
        self.vocab = vocab  # word -> id
        self.id_to_word = {v: k for k, v in vocab.items()}
        self.vocab_size = len(vocab)
    
    def encode(self, text):
        """文本 → token ids"""
        tokens = text.lower().split()
        ids = [self.vocab.get(t, self.vocab.get('<unk>')) for t in tokens]
        return ids
    
    def decode(self, ids):
        """token ids → 文本"""
        words = [self.id_to_word.get(i, '<unk>') for i in ids]
        return ' '.join(words)

# 创建一个玩具词表
vocab = {
    '<pad>': 0, '<unk>': 1, '<eos>': 2,
    'the': 3, 'cat': 4, 'sat': 5, 'on': 6, 'mat': 7,
    'dog': 8, 'ran': 9, 'in': 10, 'park': 11,
    'a': 12, 'big': 13, 'small': 14, 'happy': 15
}

tokenizer = SimpleTokenizer(vocab)

# 测试
text = "the cat sat on the mat"
ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)
print(f"原文: '{text}'")
print(f"编码: {ids}")
print(f"解码: '{decoded}'")
assert text == decoded, "编码解码不一致!"
print("✓ Tokenizer 正确")


3. 使用预训练模型 (代码示例)

【注意】以下代码需要安装 transformers 库并下载模型
如果网络环境允许，可以实际运行

--- 文本分类 (情感分析) ---

from transformers import pipeline

# 最简单的方式: 使用 pipeline
classifier = pipeline("sentiment-analysis")
result = classifier("I love this tutorial! It's amazing.")
print(result)
# [{'label': 'POSITIVE', 'score': 0.9998}]

--- 文本生成 ---

from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")
result = generator("Once upon a time", max_length=50)
print(result[0]['generated_text'])

--- 更细粒度的控制 ---

from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

# 编码
input_ids = tokenizer.encode("Hello, how are", return_tensors="pt")

# 生成
output = model.generate(input_ids, max_length=20, temperature=0.7)

# 解码
text = tokenizer.decode(output[0])
print(text)


--- 用我们的 MiniGPT 模拟 Hugging Face 的使用流程 ---
原文: 'the cat sat on the mat'
编码: [3, 4, 5, 6, 3, 7]
解码: 'the cat sat on

## 4. 微调 (Fine-tuning) 的概念


微调的本质是在已有通用能力上继续训练，让模型适应特定任务或领域。数据越少，越要谨慎防止过拟合。

冻结部分层和 LoRA 都是在降低训练成本，同时尽量保留预训练模型已有能力。

In [6]:
print("\n" + "=" * 60)
print("4. 微调 (Fine-tuning)")
print("=" * 60)

print("""
【什么是微调？】

预训练模型已经学会了语言的通用知识。
微调 = 在你的特定任务数据上继续训练，让模型适应你的需求。

【微调的类型】

1. 全量微调 (Full Fine-tuning):
   更新所有参数
   - 效果最好
   - 但需要大量显存和计算

2. 冻结部分层:
   只更新最后几层
   - 更快，显存更少
   - 底层的通用知识保持不变

3. LoRA (Low-Rank Adaptation):
   只训练小的附加矩阵
   - 参数效率极高
   - 现在最流行的方法！
   
   原理: W_new = W_old + A × B
   W_old: 冻结的原始权重 (d×d)
   A: (d×r), B: (r×d), r << d
   只训练 A 和 B，参数量从 d² 减少到 2dr

4. Prompt Tuning:
   只优化输入的 prompt 向量
   - 模型完全冻结
   - 参数量最少
""")

# 演示微调过程
print("\n--- 微调演示 ---")

class SimpleClassifier(nn.Module):
    """在预训练模型上加分类头"""
    
    def __init__(self, pretrained_model, d_model, n_classes):
        super().__init__()
        self.backbone = pretrained_model  # 预训练的 Transformer
        self.classifier = nn.Linear(d_model, n_classes)  # 新加的分类头
    
    def forward(self, input_ids):
        # 通过预训练模型获取表示
        features = self.backbone(input_ids)  # (batch, seq, d_model) 这里是 logits 但我们模拟
        # 取 [CLS] 位置 (第一个 token) 的表示做分类
        cls_feature = features[:, 0, :]  # 简化: 用第一个 token
        logits = self.classifier(cls_feature)
        return logits

print("""
微调步骤:
1. 加载预训练模型
2. 冻结或部分冻结模型参数
3. 添加任务特定的头（如分类头）
4. 在任务数据上训练
5. 评估

代码示例 (Hugging Face):

  from transformers import AutoModelForSequenceClassification, Trainer
  
  # 加载预训练模型 + 分类头
  model = AutoModelForSequenceClassification.from_pretrained(
      "bert-base-uncased", 
      num_labels=2  # 二分类
  )
  
  # 冻结底层（可选）
  for param in model.bert.encoder.layer[:8].parameters():
      param.requires_grad = False
  
  # 训练
  trainer = Trainer(model=model, train_dataset=train_dataset, ...)
  trainer.train()
""")


4. 微调 (Fine-tuning)

【什么是微调？】

预训练模型已经学会了语言的通用知识。
微调 = 在你的特定任务数据上继续训练，让模型适应你的需求。

【微调的类型】

1. 全量微调 (Full Fine-tuning):
   更新所有参数
   - 效果最好
   - 但需要大量显存和计算

2. 冻结部分层:
   只更新最后几层
   - 更快，显存更少
   - 底层的通用知识保持不变

3. LoRA (Low-Rank Adaptation):
   只训练小的附加矩阵
   - 参数效率极高
   - 现在最流行的方法！

   原理: W_new = W_old + A × B
   W_old: 冻结的原始权重 (d×d)
   A: (d×r), B: (r×d), r << d
   只训练 A 和 B，参数量从 d² 减少到 2dr

4. Prompt Tuning:
   只优化输入的 prompt 向量
   - 模型完全冻结
   - 参数量最少


--- 微调演示 ---

微调步骤:
1. 加载预训练模型
2. 冻结或部分冻结模型参数
3. 添加任务特定的头（如分类头）
4. 在任务数据上训练
5. 评估

代码示例 (Hugging Face):

  from transformers import AutoModelForSequenceClassification, Trainer

  # 加载预训练模型 + 分类头
  model = AutoModelForSequenceClassification.from_pretrained(
      "bert-base-uncased", 
      num_labels=2  # 二分类
  )

  # 冻结底层（可选）
  for param in model.bert.encoder.layer[:8].parameters():
      param.requires_grad = False

  # 训练
  trainer = Trainer(model=model, train_dataset=train_dataset, ...)
  trainer.train()



## 5. LoRA 的简单实现


LoRA 把权重更新限制在低秩矩阵 $A B$ 中，只训练少量新增参数，冻结原始大模型权重。

这使得大模型适配新任务时显存更低、训练更快，也更方便保存和切换多个任务适配器。

In [7]:
print("\n" + "=" * 60)
print("5. LoRA 简单实现")
print("=" * 60)

class LoRALayer(nn.Module):
    """LoRA: Low-Rank Adaptation"""
    
    def __init__(self, original_layer, rank=4):
        super().__init__()
        d_in = original_layer.in_features
        d_out = original_layer.out_features
        
        # 冻结原始权重
        self.original = original_layer
        for param in self.original.parameters():
            param.requires_grad = False
        
        # 低秩分解矩阵
        self.A = nn.Parameter(torch.randn(d_in, rank) * 0.01)
        self.B = nn.Parameter(torch.zeros(rank, d_out))
        # B 初始化为 0，所以初始时 LoRA 不改变原始行为
    
    def forward(self, x):
        # W_new = W_original + A × B
        original_output = self.original(x)
        lora_output = x @ self.A @ self.B
        return original_output + lora_output

# 演示 LoRA
print("--- LoRA 参数效率 ---")
d_model = 512
original = nn.Linear(d_model, d_model)
lora = LoRALayer(original, rank=8)

original_params = d_model * d_model
lora_params = d_model * 8 + 8 * d_model
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)

print(f"原始层参数: {original_params:,} ({d_model}×{d_model})")
print(f"LoRA 可训练参数: {trainable:,} ({d_model}×8 + 8×{d_model})")
print(f"参数比例: {trainable/original_params*100:.1f}%")
print(f"\n→ 只需训练 {trainable/original_params*100:.1f}% 的参数就能适应新任务！")

# 验证初始时 LoRA 不改变输出
x = torch.randn(1, d_model)
with torch.no_grad():
    orig_out = original(x)
    lora_out = lora(x)
    diff = (orig_out - lora_out).abs().max().item()
print(f"\n初始时 LoRA 输出与原始差异: {diff:.10f}")
print("✓ B 初始化为 0，所以 LoRA 初始不改变模型行为")


5. LoRA 简单实现
--- LoRA 参数效率 ---
原始层参数: 262,144 (512×512)
LoRA 可训练参数: 8,192 (512×8 + 8×512)
参数比例: 3.1%

→ 只需训练 3.1% 的参数就能适应新任务！

初始时 LoRA 输出与原始差异: 0.0000000000
✓ B 初始化为 0，所以 LoRA 初始不改变模型行为


## 6. 总结和学习路线图


这一节把前面所有知识连接到真实 LLM 工程：词嵌入、注意力、Transformer、预训练、微调和部署会共同出现在实际项目中。

建议用一个小项目串起来，例如：加载中文情感分类数据集，使用 Hugging Face 模型做推理，再尝试 LoRA 微调。

In [8]:
print("\n" + "=" * 60)
print("6. 总结和下一步学习路线")
print("=" * 60)

print("""
🎉 恭喜！你已经完成了从零开始的 AI 学习之旅！

【回顾我们学到了什么】

第1章 数学基础:
  ✓ 线性代数 (向量、矩阵、特征值)
  ✓ 微积分 (导数、梯度、反向传播)
  ✓ 概率统计 (贝叶斯、分布、信息论)
  ✓ 优化方法 (SGD、Adam)

第2章 Python 工具:
  ✓ NumPy (高效数值计算)
  ✓ Pandas (数据处理)
  ✓ Matplotlib (可视化)

第3章 机器学习:
  ✓ 线性回归 & 逻辑回归
  ✓ 决策树 & 随机森林
  ✓ 模型评估 (交叉验证、偏差-方差)

第4章 深度学习:
  ✓ 从零实现神经网络
  ✓ PyTorch 框架
  ✓ CNN (卷积神经网络)
  ✓ RNN/LSTM (循环神经网络)

第5章 大语言模型:
  ✓ 词嵌入 (Word2Vec)
  ✓ 注意力机制
  ✓ Transformer 架构
  ✓ 预训练模型和微调

【下一步学习建议】

1. 实践项目:
   - 用 Hugging Face 做一个文本分类项目
   - 用 PyTorch 训练一个小型 CNN 做图像分类
   - 尝试微调一个小型语言模型

2. 深入学习:
   - 读 "Attention Is All You Need" 论文
   - 学习 RLHF (人类反馈强化学习)
   - 了解 RAG (检索增强生成)
   - 学习 Agent 和 Tool Use

3. 工程实践:
   - 模型部署 (ONNX, TensorRT)
   - 分布式训练
   - 模型量化和加速

4. 推荐资源:
   - fast.ai 课程 (实践导向)
   - Stanford CS229/CS231n/CS224n
   - Andrej Karpathy 的 YouTube 频道
   - Hugging Face 官方教程
""")


6. 总结和下一步学习路线

🎉 恭喜！你已经完成了从零开始的 AI 学习之旅！

【回顾我们学到了什么】

第1章 数学基础:
  ✓ 线性代数 (向量、矩阵、特征值)
  ✓ 微积分 (导数、梯度、反向传播)
  ✓ 概率统计 (贝叶斯、分布、信息论)
  ✓ 优化方法 (SGD、Adam)

第2章 Python 工具:
  ✓ NumPy (高效数值计算)
  ✓ Pandas (数据处理)
  ✓ Matplotlib (可视化)

第3章 机器学习:
  ✓ 线性回归 & 逻辑回归
  ✓ 决策树 & 随机森林
  ✓ 模型评估 (交叉验证、偏差-方差)

第4章 深度学习:
  ✓ 从零实现神经网络
  ✓ PyTorch 框架
  ✓ CNN (卷积神经网络)
  ✓ RNN/LSTM (循环神经网络)

第5章 大语言模型:
  ✓ 词嵌入 (Word2Vec)
  ✓ 注意力机制
  ✓ Transformer 架构
  ✓ 预训练模型和微调

【下一步学习建议】

1. 实践项目:
   - 用 Hugging Face 做一个文本分类项目
   - 用 PyTorch 训练一个小型 CNN 做图像分类
   - 尝试微调一个小型语言模型

2. 深入学习:
   - 读 "Attention Is All You Need" 论文
   - 学习 RLHF (人类反馈强化学习)
   - 了解 RAG (检索增强生成)
   - 学习 Agent 和 Tool Use

3. 工程实践:
   - 模型部署 (ONNX, TensorRT)
   - 分布式训练
   - 模型量化和加速

4. 推荐资源:
   - fast.ai 课程 (实践导向)
   - Stanford CS229/CS231n/CS224n
   - Andrej Karpathy 的 YouTube 频道
   - Hugging Face 官方教程



## 学习检查：预训练模型应该掌握什么

你应该能回答：

- Tokenizer 的输出为什么是 token id 而不是词向量？
- `pipeline` 和 `AutoModel` 的区别是什么？
- 微调和从零训练相比，节省了什么？
- LoRA 中 rank 越小意味着什么取舍？
- 选择 Hugging Face 模型时需要看哪些信息？

## 实战建议

1. 先用 pipeline 跑通任务，确认模型能力和输入输出格式。
2. 再切换到 tokenizer + model，自己控制 batch、padding、truncation。
3. 小数据集先冻结 backbone 或使用 LoRA，避免全量微调过拟合。
4. 记录模型版本、数据版本、随机种子和评估指标，保证实验可复现。
5. 部署前关注推理延迟、显存占用、量化方式和许可证限制。

## 深入理解：为什么实际项目优先用预训练模型

从零训练语言模型需要海量数据、算力和工程经验。预训练模型已经在大规模语料上学习了通用语言能力，实际项目通常只需要把这种能力迁移到具体任务。

这就是“预训练 + 适配”的思想：预训练阶段学习通用能力，适配阶段学习任务格式、领域术语或输出风格。适配可以是 prompt、少量样本、全量微调、LoRA 或其它参数高效微调方法。

对初学者来说，最实用的路径是先用 pipeline 快速跑通，再逐步深入 tokenizer、模型 forward、训练循环和微调配置。

## Tokenizer 是模型边界的一部分

Tokenizer 不是可有可无的预处理脚本，而是模型能力边界的一部分。模型训练时使用什么 tokenizer，推理时就必须使用同一个 tokenizer 或兼容 tokenizer。

分词方式会影响序列长度、未知词处理、多语言效果和推理成本。同一句话被切成更少 token，通常推理更快；但词表过大又会增加 embedding 和输出层参数。

在真实项目中，遇到奇怪输出时，检查 tokenizer 往往很有价值：看看输入被切成了哪些 token，是否被截断，特殊 token 是否添加正确，padding 和 attention mask 是否匹配。

## LoRA 的取舍

LoRA 的优势是参数少、显存低、便于保存多个任务适配器。它适合在已有大模型上学习特定任务、风格或领域知识。

rank 越大，可训练参数越多，表达能力越强，但显存和过拟合风险也更高。rank 太小则可能无法充分适配任务。实际项目中通常会把 rank、学习率、目标模块、训练轮数一起调参。

LoRA 不会神奇地弥补数据质量问题。如果微调数据噪声很大、格式混乱或目标不一致，模型仍然会学到不稳定行为。微调前整理高质量数据，往往比盲目增大 rank 更重要。

## 项目注意事项

预训练模型上线前，还要检查隐私数据、输出安全性、成本预算和失败兜底策略。模型能跑通 demo 只是第一步，能稳定、合规、可监控地服务真实用户，才是工程落地。

## 核心术语对照

- **Pretraining**：在大规模通用语料上训练模型，学习语言基础能力。
- **Inference**：使用训练好的模型做预测或生成，不更新参数。
- **Fine-tuning**：在特定任务数据上继续训练模型，使其适应目标任务。
- **Full Fine-tuning**：更新全部参数，效果强但成本高。
- **Parameter-Efficient Fine-tuning**：只训练少量新增参数或部分参数，例如 LoRA。
- **Model Card**：模型说明文档，包含用途、限制、许可证和示例。

实际工作中，先推理验证，再决定是否微调；先读 model card，再决定是否适合你的任务和部署场景。

## 课后练习：从预训练模型走向真实项目

建议你选一个小任务，例如情感分类、标题生成或相似问题匹配。先用 Hugging Face pipeline 直接推理，记录模型在哪些样本上表现好、哪些样本上失败。然后查看 tokenizer 输出，确认文本是否被合理切分、是否被截断。

如果直接推理效果不够，再考虑少量标注数据和 LoRA 微调。微调前先写清楚评估集和指标，否则很容易只看到训练 loss 下降，却不知道模型在真实任务上是否变好。

## 选择预训练模型时看什么

实际项目中，选模型不能只看排行榜。至少要检查：

- 模型任务类型：文本分类、生成、embedding、rerank、翻译等。
- 语言覆盖：是否支持中文，中文效果是否经过验证。
- 参数规模：越大不一定越适合，成本和延迟也会增加。
- 上下文长度：能否容纳你的输入文档。
- 许可证：是否允许商用或再分发。
- 模型卡片：训练数据、适用场景、限制和示例。

Hugging Face Model Hub 的 model card 是非常重要的信息源。初学者要养成先读 model card，再写代码的习惯。

## 从 demo 到项目的落地流程

一个实用流程可以是：

1. 用 pipeline 快速验证任务是否可行。
2. 收集少量真实样本，人工检查模型输出。
3. 写 tokenizer + model 的显式推理代码，控制 batch、padding 和 truncation。
4. 设定评估指标，例如 accuracy、F1、BLEU、ROUGE 或人工评分标准。
5. 如果零样本/少样本不够，再考虑微调或 LoRA。
6. 部署前测试延迟、显存、并发、异常输入和许可证。

这样做可以避免一开始就陷入复杂训练。很多业务问题并不需要微调，好的提示、检索增强或更合适的模型选择就能解决。